In [0]:
dbutils.widgets.text("p_batch_id", "")
v_batch_id = dbutils.widgets.get("p_batch_id")

In [0]:
%run ../00-common/01.Environment-config

In [0]:
%run ../00-common/02.bronze-helpers

In [0]:
source_file = f'{landing_folder_path}/{v_batch_id}/circuits.csv'
table_name = f'{catalog_name}.{bronze_schema}.circuits'

### Step 1 - Read the csv files using dataframe reader api

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType

circuits_schema = StructType([
    StructField('circuitId', StringType()),
    StructField('url', StringType()),
    StructField('circuitName', StringType()),
    StructField('lat', DoubleType()),
    StructField('long', DoubleType()),
    StructField('locality', StringType()),
    StructField('country', StringType())
])

In [0]:
circuits_df = (
    spark.read
    .format('csv')
    .option('header','true')
    .schema(circuits_schema)
    .option('mode','FAILFAST')
    .load(source_file)
)

In [0]:
display(circuits_df)
        

### Step 2 - Add metadata

In [0]:
circuits_final_df=add_ingestion_metadata(circuits_df)


In [0]:
display(circuits_final_df)

### Step 3 - Write to bronze delta table 

In [0]:
write_to_bronze(
    input_df = circuits_final_df,
    target_table = table_name,
    batch_id = v_batch_id
)

In [0]:
display(spark.table(table_name))